# Parakeet Singlish v2 — Singapore English / Singlish ASR on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Parakeet-Singlish-v2-asr.ipynb)

[`Scrya/parakeet-tdt-0.6b-singlish-v2`](https://huggingface.co/Scrya/parakeet-tdt-0.6b-singlish-v2) is Scrya's **0.6B** Parakeet-TDT fine-tune of NVIDIA's [`parakeet-tdt-0.6b-v3`](https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3) for **Singapore English / Singlish**. [CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/), public (not gated). Hub: `transformers` + `safetensors`, architecture `ParakeetForTDT` (`model_type: parakeet_tdt`). Hub reports **627.1M** parameters. `model.safetensors` is **2.4 GB** float32 on the card (the `.nemo` file is 2.51 GB).

**Why this model.** The card finetuned the base model on **1,386 hours** of conversational Singapore English from five parts of IMDA's National Speech Corpus (NSC Parts 3–6), with meeting-room augmentations (office noise, background talkers, room echo). Output is **punctuated English**; numbers are spelled out as words (e.g. "twenty five" instead of "25"). This is version 2: the average of two training runs, then minimum word-error-rate training. On the card's 66,279 NSC test clips, normalised WER is **9.16%** over all five parts (formatted WER **15.79%**). Those are the card's numbers, not a T4 measurement from this notebook.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/Parakeet-Singlish-v2-asr.md) for when this notebook is a good fit and when it is not.

This notebook uses the **🤗 Transformers** path the card documents (lightest Colab install). GGUF files for NeMo-Speech.cpp live in **separate** repos ([f16](https://huggingface.co/Scrya/parakeet-tdt-0.6b-singlish-v2-nemo-speech-cpp-f16), [q8_0](https://huggingface.co/Scrya/parakeet-tdt-0.6b-singlish-v2-nemo-speech-cpp-q8_0)), not in this model repo.

```python
from transformers import AutoProcessor, ParakeetForTDT

repo = "Scrya/parakeet-tdt-0.6b-singlish-v2"
processor = AutoProcessor.from_pretrained(repo)
model = ParakeetForTDT.from_pretrained(repo, dtype=torch.float32).eval()
```

`config.json` records `transformers_version: 5.22.1`. Needs **Transformers 5.22 or newer**. The card decoded in **fp32** because greedy decoding in bf16 gave slightly different text for different batch sizes. A free T4 has **no native bfloat16**, so this notebook loads **float16**. Set `DTYPE = torch.float32` if a transcript looks degraded.

**Timestamps.** Transformers Parakeet TDT can return **token** timestamps via `processor.decode(..., durations=output.durations)` (same API as [`nvidia/parakeet-tdt-0.6b-v3`](https://huggingface.co/docs/transformers/model_doc/parakeet)). Word/segment timestamps are the NeMo `transcribe(..., timestamps=True)` path; installing `nemo_toolkit[asr]` is a heavy Colab change, so this notebook stays on Transformers.

**Looping caveat (card).** `max_symbols_per_step` is 3 in the config. In transformers 5.22.1, greedy TDT decoding ignores that setting. On rare clips the model can repeat a phrase many times. The card: use the NeMo file or a GGUF file for exact results.

**Licence:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). If you use the model, credit **Scrya** for the finetuned model and **NVIDIA Parakeet-TDT-0.6B-v3** for the base model.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~16 GB of VRAM. First run downloads about **2.4 GB**.

**Not in this notebook:** NeMo `.nemo` restore, NeMo-Speech.cpp GGUF, training.

**Links:** [model card](https://huggingface.co/Scrya/parakeet-tdt-0.6b-singlish-v2) · [announcement](https://x.com/testdanrun/status/2126149023539122644) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Parakeet-Singlish-v2-asr.md) · [base model](https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3)


In [ ]:
%pip install -q -U "transformers>=5.22.0" soundfile requests

# Card Transformers install is AutoProcessor + ParakeetForTDT (config: transformers 5.22.1).
# Do NOT pip install torch here. Colab already ships a CUDA wheel; forcing a
# reinstall often drops GPU support or pulls a CPU-only build.
# soundfile reads the public wav/flac clips. huggingface_hub ships with transformers.
# requests fetches one SASRBench row from the dataset viewer API.
# Use %pip (not python -I). Isolated mode drops Colab's site-packages.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.

## 1. Check the GPU

Confirm CUDA is available, print the device and free VRAM, and pick a dtype. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses. Fall back to float32 if outputs degrade (the card scored in fp32).

In [ ]:
import os
import time

import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.22.0"), (
    f"transformers {transformers.__version__} is loaded; Parakeet Singlish v2 needs 5.22+ "
    "(card: ParakeetForTDT, transformers_version 5.22.1). "
    "Run the install cell, then Runtime > Restart session."
)

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda"
# T4 (sm_75) has no native bfloat16. Checkpoint dtype is float32.
# float16 is the T4 path. Set DTYPE = torch.float32 if transcripts look degraded
# (the card decoded in fp32).
DTYPE = torch.float16
MODEL_ID = "Scrya/parakeet-tdt-0.6b-singlish-v2"
TARGET_SR = 16_000

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}  (saved weights are float32; T4 has no native bf16)")
print(f"VRAM free: {free_b / 1024**3:.1f} GB of {total_b / 1024**3:.1f} GB")
if major < 8:
    print("No native bfloat16 on this GPU — staying on float16.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB | "
        f"free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 2. Load the model

The card's call is `ParakeetForTDT.from_pretrained(repo, dtype=torch.float32)`. We only change `dtype` so the T4 runs float16 instead of defaulting to float32, then move the model to CUDA. First run downloads about **2.4 GB** (`model.safetensors`). No `trust_remote_code` — `config.json` names native `ParakeetForTDT`.

In [ ]:
from transformers import AutoProcessor, ParakeetForTDT

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = ParakeetForTDT.from_pretrained(MODEL_ID, dtype=DTYPE).eval()
model.to(DEVICE)
torch.cuda.synchronize()
print(f"loaded {MODEL_ID} in {time.perf_counter() - t0:.1f}s")
print("class:", type(model).__name__)
print("dtype:", next(model.parameters()).dtype, "| device:", next(model.parameters()).device)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.1f}M  (Hub: 627.1M)")
print("max_symbols_per_step:", getattr(model.config, "max_symbols_per_step", None))
print_vram("after load")
print()
print("If greedy transcripts later look empty, garbled, or loop, reload with DTYPE = torch.float32.")
print("Card: transformers 5.22.1 greedy TDT decoding ignores max_symbols_per_step.")


## 3. Demo 1 — public English clip

`librispeech_mr_quilter.wav` from [`hf-internal-testing/dummy-audio-samples`](https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples) (public LibriSpeech-style read speech; the same "Mr Quilter" sentence NVIDIA's Parakeet docs use). Fallback: Hugging Face CDN `sample1.flac`. This is **not** Singapore English.

The generate call matches the card: `model.generate(**inputs, return_dict_in_generate=True)` then `processor.batch_decode`.

In [ ]:
from pathlib import Path

import numpy as np
import requests
import soundfile as sf
from huggingface_hub import hf_hub_download
from IPython.display import Audio as play, display

SAMPLE_DIR = Path("parakeet_singlish_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}


def resample_mono(audio, orig_sr, target_sr=TARGET_SR):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=-1)
    if int(orig_sr) == int(target_sr) or len(audio) == 0:
        return audio
    n = int(round(len(audio) * target_sr / orig_sr))
    x_old = np.linspace(0.0, 1.0, num=len(audio), endpoint=False)
    x_new = np.linspace(0.0, 1.0, num=n, endpoint=False)
    return np.interp(x_new, x_old, audio).astype(np.float32)


def write_wav(path, array, sampling_rate):
    path = Path(path)
    audio = resample_mono(array, sampling_rate, TARGET_SR)
    sf.write(path, audio, TARGET_SR)
    return path, len(audio) / float(TARGET_SR)


def transcribe_array(audio, sampling_rate, with_timestamps=False):
    audio = resample_mono(audio, sampling_rate, TARGET_SR)
    inputs = processor(audio=[audio], sampling_rate=TARGET_SR)
    inputs = inputs.to(model.device, dtype=model.dtype)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        # max_new_tokens is not on the card; transformers otherwise uses a
        # model-agnostic max_length of 222, which can truncate a 20 s clip,
        # and the card says greedy TDT can loop when max_symbols is ignored.
        out = model.generate(**inputs, return_dict_in_generate=True, max_new_tokens=1024)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    text = processor.batch_decode(out.sequences, skip_special_tokens=True)[0]
    timestamps = None
    if with_timestamps:
        decoded = processor.decode(
            out.sequences,
            durations=out.durations,
            skip_special_tokens=True,
        )
        timestamps = decoded[1] if isinstance(decoded, tuple) and len(decoded) > 1 else decoded
    return text, elapsed, timestamps


def download_english_clip():
    try:
        src = hf_hub_download(
            repo_id="hf-internal-testing/dummy-audio-samples",
            filename="librispeech_mr_quilter.wav",
            repo_type="dataset",
        )
        audio, sr = sf.read(src)
        path, seconds = write_wav(SAMPLE_DIR / "en-mr-quilter.wav", audio, sr)
        return path, seconds, "hf-internal-testing/dummy-audio-samples/librispeech_mr_quilter.wav"
    except Exception as exc:
        print(f"Hub dummy wav failed ({type(exc).__name__}: {exc}); trying CDN sample1.flac")
        url = "https://cdn-media.huggingface.co/speech_samples/sample1.flac"
        raw = requests.get(url, headers=UA, timeout=120)
        raw.raise_for_status()
        flac_path = SAMPLE_DIR / "en-sample1.flac"
        flac_path.write_bytes(raw.content)
        audio, sr = sf.read(flac_path)
        path, seconds = write_wav(SAMPLE_DIR / "en-sample1.wav", audio, sr)
        return path, seconds, url


en_path, en_seconds, en_source = download_english_clip()
en_audio, en_sr = sf.read(en_path)
print(f"source: {en_source}")
print(f"audio: {en_seconds:.2f}s  |  sr={TARGET_SR}")
print("reference (Mr Quilter clip): mister quilter is the apostle of the middle classes...")
display(play(str(en_path)))

en_text, en_elapsed, _ = transcribe_array(en_audio, en_sr)
print()
print("transcript:", en_text)
print(f"wall time: {en_elapsed:.2f}s  |  audio: {en_seconds:.2f}s")
print_vram("after English clip")


## 4. Demo 2 — Singapore English / Singlish clip

One test clip from [`mjwong/SASRBench-v1`](https://huggingface.co/datasets/mjwong/SASRBench-v1), a public Singlish ASR benchmark (Hub license **MIT**). The dataset card: segments come from IMDA NSC **Part 3 Same Room** close-talk recordings (Singapore English conversation). That is the same corpus family this model was finetuned on; this notebook uses a published Hub test clip, not an official IMDA download.

Fetched through the Hugging Face dataset viewer API (one row, no 1.6 GB parquet dump). If that request fails, the cell falls back to the English clip from Demo 1 and says so.

The SASRBench reference keeps discourse particles in brackets (`[ah]`, `[leh]`) and spaced acronyms (`n_t_u`). The model writes punctuated English, so compare words, not formatting.

In [ ]:
SASR_OFFSET = 0  # row 0: conversational Singlish ("I going to Korea ah...")


def fetch_sasrbench_clip(offset=SASR_OFFSET):
    rows = requests.get(
        "https://datasets-server.huggingface.co/rows",
        params={
            "dataset": "mjwong/SASRBench-v1",
            "config": "default",
            "split": "test",
            "offset": offset,
            "length": 1,
        },
        headers=UA,
        timeout=120,
    )
    rows.raise_for_status()
    row = rows.json()["rows"][0]["row"]
    audio_field = row["audio"]
    src = audio_field[0]["src"] if isinstance(audio_field, list) else audio_field["src"]
    path = SAMPLE_DIR / f"sg-sasrbench-{offset}.wav"
    response = requests.get(src, headers=UA, timeout=120)
    response.raise_for_status()
    path.write_bytes(response.content)
    audio, sr = sf.read(path)
    return path, audio, sr, row.get("transcript")


sg_source = "mjwong/SASRBench-v1"
try:
    sg_path, sg_audio, sg_sr, sg_ref = fetch_sasrbench_clip()
    sg_path, sg_seconds = write_wav(sg_path, sg_audio, sg_sr)
    sg_audio, sg_sr = sf.read(sg_path)
except Exception as exc:
    print(f"SASRBench download failed ({type(exc).__name__}: {exc})")
    print("Falling back to the public English clip from Demo 1.")
    sg_source = "fallback: English LibriSpeech dummy"
    sg_path, sg_seconds = en_path, en_seconds
    sg_audio, sg_sr = sf.read(sg_path)
    sg_ref = "(fallback English clip; no SASRBench transcript)"

print(f"source: {sg_source}")
print(f"audio: {sg_seconds:.2f}s  |  sr={TARGET_SR}")
print("reference:", sg_ref)
display(play(str(sg_path)))

sg_text, sg_elapsed, _ = transcribe_array(sg_audio, sg_sr)
print()
print("transcript:", sg_text)
print(f"wall time: {sg_elapsed:.2f}s  |  audio: {sg_seconds:.2f}s")
print_vram("after Singapore English clip")


## 5. Demo 3 — token timestamps and timing

Transformers Parakeet TDT returns **token** timestamps when you pass `durations` into `processor.decode`, as in the [Parakeet docs](https://huggingface.co/docs/transformers/model_doc/parakeet). That is not the NeMo word/segment table (`hyp.timestamp["word"]`). Consecutive subword tokens are also grouped into whitespace-delimited words for reading.

Uses the Singapore English clip when Demo 2 got SASRBench, otherwise the English clip. Wall time is this T4 run, not a card benchmark.

In [ ]:
def group_token_timestamps_into_words(token_rows):
    """Join subword tokens into words. Display helper, not a card API."""
    words = []
    current = None
    for row in token_rows:
        tok = str(row.get("token", ""))
        start, end = row.get("start"), row.get("end")
        is_punct = tok.strip() in {".", ",", "?", "!", ";", ":"}
        starts_word = tok[:1].isspace() or tok[:1] in {"-", "—"}
        if current is None:
            current = {"word": tok.lstrip(), "start": start, "end": end}
            if is_punct:
                words.append(current)
                current = None
            continue
        if starts_word or is_punct:
            if current["word"]:
                words.append(current)
            current = {"word": tok.strip(), "start": start, "end": end}
            if is_punct:
                if current["word"]:
                    words.append(current)
                current = None
            continue
        current["word"] += tok
        current["end"] = end
    if current and current["word"]:
        words.append(current)
    return words


ts_audio, ts_sr = sg_audio, sg_sr
ts_seconds = sg_seconds
ts_label = sg_source

text, elapsed, ts = transcribe_array(ts_audio, ts_sr, with_timestamps=True)
rtf = elapsed / ts_seconds if ts_seconds else float("nan")

print(f"clip: {ts_label} ({ts_seconds:.2f}s)")
print("transcript:", text)
print(f"audio: {ts_seconds:.2f}s  |  wall time: {elapsed:.2f}s  |  RTF {rtf:.3f}x")
print("(RTF < 1 means faster than real time. This is T4 + short-file overhead, not a card figure.)")
print()

token_rows = ts[0] if ts and isinstance(ts, (list, tuple)) and ts and isinstance(ts[0], list) else (ts or [])
print(f"token timestamps ({len(token_rows)} tokens; Transformers Parakeet TDT API):")
for row in token_rows[:24]:
    print(f"  {row['start']:6.2f}s – {row['end']:6.2f}s  {row.get('token', '')!r}")
if len(token_rows) > 24:
    print(f"  ... {len(token_rows) - 24} more tokens")

word_rows = group_token_timestamps_into_words(token_rows)
print()
print(f"grouped word spans ({len(word_rows)} words; from token timestamps, not NeMo word timestamps):")
for row in word_rows[:20]:
    print(f"  {row['start']:6.2f}s – {row['end']:6.2f}s  {row['word']}")
if len(word_rows) > 20:
    print(f"  ... {len(word_rows) - 20} more words")

if text and len(text) > 4000:
    print()
    print("Transcript is unusually long — possible TDT loop (card: max_symbols ignored in transformers 5.22.1).")

print_vram("after timestamps")


## 6. Caveats and cleanup

**What this notebook is for.** Offline Singapore English / Singlish transcription on a free T4 with the card's 🤗 Transformers API: a public English clip, a public Singlish conversational clip, token timestamps, and wall time.

**What it is not.** NeMo `.nemo` restore (`nemo_toolkit` 2.7.3 on the card), NeMo word/segment timestamps, NeMo-Speech.cpp GGUF, streaming, speech translation, or speaker labels. Pair with [Nemotron-3 Diarization](Nemotron-3-Diarization.ipynb) if you need who-spoke-when.

**Length.** The card: most training/test clips are under 20 seconds; split long audio first.

**Languages.** Finetuned on English only. Code-switched Mandarin / Malay / other speech is written as English or left out. The 24 other European languages of the base model were not re-tested after finetuning.

**T4 notes.** float16 weights on T4; no bfloat16. The card scored in fp32. Keep single clips short; for long recordings, split first.

**Related notebooks:** [Fluid Speech Pro](Fluid-Speech-Pro-multilingual-asr.ipynb) (same Parakeet-TDT-0.6B-v3 lineage, 25 European languages), [Nemotron-Labs Unified ASR 0.6B](Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb), [Rumble-2-Flash](Rumble-2-Flash-noisy-speech-to-text.ipynb), [Granite Speech 5.0 2B](granite-speech-5.0-2b-asr-translation.ipynb), [oruk-loop-153M](oruk-loop-153M-looped-asr.ipynb), [Whisper](AIBackends-Whisper-audio-transcription.ipynb).

Free the GPU before you open another notebook:

In [ ]:
import gc

print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del model, processor
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
